# Lab 16: The Ventriloquist's Secret

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-16.ipynb)

**What you will do:** test how far you can separate a voice from a moving mouth before the ventriloquism effect breaks down, compare your breakdown point with the range Lab 16 describes, and run a Bayesian cue-combination model that shows why vision normally wins the argument over where a sound came from.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 16 you watched a friend mouth words while a speaker to their side played the voice,
and heard the sound as coming from their moving lips, not from the speaker. This is the
**ventriloquism effect**: when vision and hearing disagree about where something is, the
brain trusts the more spatially precise sense, vision, and the disagreement is only
registered once the speaker is moved far enough away that the two senses can no longer be
plausibly describing the same event.

## Record your results

Repeat Step 4: move the speaker further from your friend in steps, and for each separation
(in degrees of visual angle, estimated by eye, or in centimetres at a fixed distance) note
whether you still perceived the sound as coming from your friend's mouth.

In [ ]:
# example data: replace with your own trials
trials = pd.DataFrame({
    "separation_deg": [5, 10, 15, 20, 25, 30, 35, 40, 45],
    "heard_from_mouth": [True, True, True, True, True, False, False, False, False],
})
display(trials)
trials["heard_from_mouth"].astype(int).plot(marker="o", figsize=(6, 3))
plt.xticks(range(len(trials)), trials["separation_deg"]); plt.yticks([0, 1], ["speaker", "mouth"])
plt.xlabel("separation (degrees)"); plt.ylabel("perceived source"); plt.show()

## Compare

Lab 16 says the illusion weakens as the gap widens and "somewhere past a few tens of degrees
of separation, most people start to hear the sound coming from the speaker's actual
location." Find your own breakdown point and see whether it falls in that same rough range.

In [ ]:
switched = trials.loc[~trials["heard_from_mouth"], "separation_deg"]
if len(switched):
    breakdown = switched.min()
    lower = trials.loc[trials.separation_deg < breakdown, "separation_deg"].max()
    print(f"Your illusion broke down somewhere between {lower} and {breakdown} degrees.")
    if 20 <= breakdown <= 60:
        print("This sits within the lab's 'a few tens of degrees' description.")
    else:
        print("This falls outside a loose reading of 'a few tens of degrees'. The lab does not give an"
              " exact number, and the true breakdown point varies with room acoustics, how directional"
              " the speaker is, and how attentively you are listening, so treat this as a rough estimate.")
else:
    print("The illusion held at every separation you tried -- extend the range further to find where it breaks.")

## The AI side

The lab explains the ventriloquism effect as **Bayesian sensor fusion**: each sense
estimates the sound's location with its own uncertainty, and the brain weights each
estimate by how reliable (precise) it is. The code below simulates exactly this for two
noisy location estimates, a precise "visual" one and an imprecise "auditory" one, and
combines them the way the text says the brain does.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np

rng = np.random.default_rng(2)
true_location = 0.0   # degrees, arbitrary reference point

vision_precision = 1 / 2.0 ** 2     # low variance = high precision (vision localises well)
audio_precision = 1 / 8.0 ** 2      # high variance = low precision (hearing localises poorly)

n_trials = 2000
vision_estimate = rng.normal(true_location, 2.0, n_trials)
audio_estimate = rng.normal(true_location, 8.0, n_trials)

# Bayes-optimal combination: weight each estimate by its precision
combined = (vision_precision * vision_estimate + audio_precision * audio_estimate) / (vision_precision + audio_precision)
combined_precision = vision_precision + audio_precision

df = pd.DataFrame({"vision alone": vision_estimate, "audio alone": audio_estimate, "combined estimate": combined})
display(df.describe().loc[["mean", "std"]].round(2))
print(f"Predicted combined standard deviation: {1 / np.sqrt(combined_precision):.2f} degrees "
      f"(observed: {combined.std():.2f})")

df[["vision alone", "audio alone", "combined estimate"]].plot.hist(bins=40, alpha=0.5, figsize=(7, 3.5))
plt.xlabel("estimated location (degrees from true source)"); plt.show()

The combined estimate lands much closer to vision's precision than to a simple average of
the two, because vision is the more reliable sense for location, the Bayesian combination
leans heavily towards it, just as your own perception leaned towards the moving mouth. The
disanalogy: this model always combines the two estimates smoothly; it has no equivalent of
the sudden "these must be two different events" switch you experienced once the speaker
moved far enough away, which is closer to how the brain actually behaves at large
discrepancies.

## Pool the class data

Pool the class's breakdown separations: the smallest separation, in degrees, at which each
person first heard the sound from the speaker rather than the mouth.

In [ ]:
import io
csv_text = """id,breakdown_deg
P01,30
P02,45
P03,25
P04,50
P05,35
P06,40
"""  # example data: replace with your class CSV, e.g. pd.read_csv("ventriloquism_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
boot = [class_df["breakdown_deg"].sample(len(class_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean breakdown separation: {class_df['breakdown_deg'].mean():.0f} degrees "
      f"(95% bootstrap interval {lo:.0f}-{hi:.0f}, n = {len(class_df)})")
class_df["breakdown_deg"].plot.hist(bins=6, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.xlabel("breakdown separation (degrees)"); plt.show()

## Questions to think about

1. In the Bayesian simulation, what happens to the combined estimate if you make audio_estimate's standard deviation much larger, say 20 degrees, while leaving vision at 2 degrees? Does the combined estimate move closer to vision or further from it, and does this match the claim that "the brain combines the estimates in proportion to their precision"?
2. The lab notes that in a dark room, auditory localisation is weighted more heavily because vision has become unreliable. How would you change the simulation's vision_precision to represent a dark room, and what would you expect to happen to the combined estimate?
3. The text distinguishes "where" questions (vision dominates) from "when" questions (hearing dominates), linking this to Lab 15's finding that hearing has much finer temporal resolution than vision. Why might it make evolutionary sense for each sense to specialise, rather than both senses being equally good at both jobs?
4. The AI Connection describes "misassociation" in self-driving cars -- wrongly binding a radar return from one vehicle to a camera image of a different one. What would the equivalent misassociation look like in the ventriloquism set-up, if there were two friends mouthing words and two speakers?

## Challenge

Try Step 3 with a dubbed film in a language you do not speak. Note how large the mismatch
between the lip movements and the audio is, and rate on a 0-5 scale how strongly you still
perceived the voice as coming from the actor's mouth. Compare this rating with your breakdown
separation from the live speaker-and-friend version above: does a much larger, more obviously
wrong mismatch still capture your perception, or does the illusion feel weaker on screen?

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-16.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")